# Fast learners (Project 17) - Seminar v4

**Question:** if networks are selected only by their validation accuracy after a *short* training budget,
does the search find networks that learn fast - and do they pay for it (lower final accuracy, less robust)?

## Run order
**Step 1 - benchmark (once, on the GPU):** run *Setup → Load Data → Training → Performance benchmark*.
Read where "ms per network" stops dropping and set `MAX_PARALLEL` in Setup to that value.

**Step 2 - experiment (Rocket or Colab GPU):** set `QUICK = False`, `TEST_RUN = False`, `RUN_BENCHMARK = False`, then *Run all*.
At the end, everything is saved to `results_<mode>_<date>.json`.

**Step 3 - analysis only (anyone, no GPU needed):** upload the `.json`, set `LOAD_RESULTS` in Setup,
run *Setup → Load results → everything below it*. Skip the cells in between (they train networks).

## What changed vs Seminar (3)
- `NEW`: run order + an explanation before every section
- `CHANGED`: runs on Rocket too - pip only if packages are missing, no TensorFlow/Flax needed on Rocket
- `NEW`: `XLA_PYTHON_CLIENT_PREALLOCATE=false` so the benchmark shows the *real* GPU memory use
- `NEW`: `RUN_BENCHMARK` switch, so *Run all* doesn't repeat the benchmark every time
- `CHANGED`: the robustness test (corrupted images) runs **before saving**, so its results are in the saved file
- `NEW`: **Save results** / **Load results** cells
- everything else (search, training, analysis) is unchanged from Seminar (3)

# **Install**
Only installs when something is missing (Colab). On Rocket everything is already in the venv.

In [ ]:
# CHANGED: only install when needed (on Rocket the compute node may have no internet)
import importlib.util
if importlib.util.find_spec("optuna") is None:
    %pip install -q --upgrade jax-ai-stack optuna

# **Setup**
All settings + helper functions. **No training happens here.**
- run size: `QUICK` / `TEST_RUN`, budgets, seeds, generations
- `MAX_PARALLEL`: how many networks go to the GPU in one call (set it from the benchmark)
- search space: the 6 hyperparameters that are searched; `decode` / `encode` translate genome ⇄ hyperparameters
- `grid_cell`: where a network lands on the 8x8 diversity grid (learning rate x number of parameters)
- `BASELINES`: hand-picked networks every winner is compared with

In [ ]:
import os, time, json, pickle
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # NEW: must be set before importing jax -> real memory use
import jax, jax.numpy as jnp
import numpy as np
import optuna
import optax
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Change below values for the full run
QUICK = True
TEST_RUN = True
RUN_BENCHMARK = True     # NEW: step 1 only; set False for the real run
LOAD_RESULTS = None      # NEW: analysis only -> e.g. "results_full_20261111-1200.json"

BUDGETS = [2_560, 12_800, 32_000, 64_000, 128_000]        # Training images seen per candidate during search
BATCH_SIZES = [32, 64, 128]
SEARCH_METHODS = ["GA", "MAP-Elites", "Optuna TPE", "Random search"]
SEEDS = [0, 1] if QUICK else list(range(10))
CPG = 24                                                  # Candidate per generation (CPG)
GENERATIONS = 5 if QUICK else 20                          # CPG * GENERATIONS - networks per search
EVAL_EVERY = 640                                          # Evaluate every 640 images
THRESHOLD = [0.75, 0.80, 0.85]                            # speed - accuracy models need to reach
MAX_PARALLEL = 24                                        # Networks trained at once on the GPU

TOP_K, RECHECK_SEEDS = 5, 3
EPOCHS = 2 if QUICK else 20
N_TRAIN = 50_000
SPLIT_SEED = 42
DATA_SEED = 1234
SGD_MOMENTUM = 0.9

# Search space - genes and their ranges
SEARCH_SPACE = {
    "lr": ("log", 1e-4, 1.0),
    "optimizer": ("choice", ["sgd", "adam"]),
    "batch_size": ("choice", BATCH_SIZES),
    "layers": ("choice", [1, 2, 3]),                    # number of hidden Dense layers
    "width": ("int", 16, 256),                          # units per hidden layer
    "activation": ("choice", ["relu", "tanh"]),
}

if TEST_RUN:
    BUDGETS, SEEDS, GENERATIONS, TOP_K, RECHECK_SEEDS = [2_560], [0], 2, 3, 1

# Baseline models
BASELINES = {
    "Adam 1e-3": dict(lr=1e-3, optimizer="adam", batch_size=128, layers=2, width=128, activation="relu"),
    "Adam 3e-4": dict(lr=3e-4, optimizer="adam", batch_size=128, layers=2, width=128, activation="relu"),
    "SGD 1e-2":  dict(lr=1e-2, optimizer="sgd",  batch_size=128, layers=2, width=128, activation="relu"),
}
N_GENES = len(SEARCH_SPACE)

def decode(genome):
    """Genome -> decode -> Hyperparameters"""
    g = np.clip(np.asarray(genome, dtype=float), 0.0, 1.0)
    hp = {}
    for x, (name, spec) in zip(g, SEARCH_SPACE.items()):
        kind = spec[0]
        if kind == "log":
            lo, hi = np.log10(spec[1]), np.log10(spec[2])
            hp[name] = float(10 ** (lo + x * (hi - lo)))
        elif kind == "int":
            hp[name] = int(round(spec[1] + x * (spec[2] - spec[1])))
        else:
            options = spec[1]
            hp[name] = options[min(int(x * len(options)), len(options) - 1)]
    return hp

def encode(hp):
    """Hyperparameters -> encode -> Genome"""
    g = []
    for name, spec in SEARCH_SPACE.items():
        kind, v = spec[0], hp[name]
        if kind == "log":
            lo, hi = np.log10(spec[1]), np.log10(spec[2])
            g.append((np.log10(v) - lo) / (hi - lo))
        elif kind == "int":
            g.append((v - spec[1]) / (spec[2] - spec[1]))
        else:
            options = spec[1]
            g.append((options.index(v) + 0.5) / len(options))
    return np.array(g)

def describe(hp):
    return (f"{hp['optimizer']:<4} lr={hp['lr']:.4f} batch={hp['batch_size']:3d} "
            f"{hp['layers']}x{hp['width']:3d} {hp['activation']}")

def n_params(hp, n_in=784, n_out=10):
    w, L = hp["width"], hp["layers"]
    return n_in * w + w + (L - 1) * (w * w + w) + w * n_out + n_out



# MAP-Elites (Learning rate * Number of network parameters)
GRID_BINS = 8
GOOD_CELL_MARGIN = 0.03
_P_MIN = n_params(dict(width=16, layers=1))
_P_MAX = n_params(dict(width=256, layers=3))

def grid_cell(hp):
    """Return the (row, col) cell of the diversity grid for one network."""
    lr_pos = (np.log10(hp["lr"]) + 4) / 4  # 0..1
    size_pos = (np.log10(n_params(hp)) - np.log10(_P_MIN)) / (np.log10(_P_MAX) - np.log10(_P_MIN))
    to_bin = lambda p: int(np.clip(p * GRID_BINS, 0, GRID_BINS - 1))
    return to_bin(lr_pos), to_bin(size_pos)

def sample_order(n_samples, seed=DATA_SEED):
    """Order of training images: identical for every network. Reshuffles each epoch,
    so budgets larger than 50,000 images work."""
    rng = np.random.default_rng(seed)
    n_epochs = -(-n_samples // N_TRAIN)
    return np.concatenate([rng.permutation(N_TRAIN) for _ in range(n_epochs)])[:n_samples]

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

# **Load Data**
Fashion-MNIST: 60k training images (split into 50k train + 10k validation, fixed seed 42) and 10k test images.
Pixels are scaled to [0, 1] and each 28x28 image is flattened to 784 numbers.

In [ ]:
#Loading data
# CHANGED: on Rocket use the file we already downloaded (no TensorFlow there); in Colab use Keras as before
path = "../data/fashion_mnist.npz"
if os.path.exists(path):
    x_train, y_train, x_test, y_test = np.load(path).values()
else:
    from tensorflow.keras.datasets import fashion_mnist
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize images to [0, 1]
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Create a fixed 50k/10k training-validation split
rng = np.random.default_rng(42)
indices = rng.permutation(len(x_train))

train_idx = indices[:50000]
val_idx = indices[50000:]

x_val, y_val = x_train[val_idx], y_train[val_idx]
x_train, y_train = x_train[train_idx], y_train[train_idx]

# Flatten 28x28 images
x_train = jnp.asarray(x_train.reshape(-1, 784))
x_val = jnp.asarray(x_val.reshape(-1, 784))
x_test = jnp.asarray(x_test.reshape(-1, 784))

y_train = jnp.asarray(y_train)
y_val = jnp.asarray(y_val)
y_test = jnp.asarray(y_test)

print("Training:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

In [ ]:
#Show image
plt.imshow(x_train[0].reshape(28, 28), cmap="gray")
plt.title(f"Label: {y_train[0]}")
plt.axis("off")
plt.show()

# **Training**
The engine: trains many networks **at once** on the GPU.
- Every network is built at the **maximum size** (3 layers x 256 units); unused units/layers are masked out.
  That way all networks have the same shape, so `jax.vmap` can train them side by side and `jax.jit` compiles it once.
- `train_population(hps, budget, seed)` trains a list of recipes on `budget` images.
  Networks are grouped by batch size and sent to the GPU in chunks of `MAX_PARALLEL`
  (padded to a multiple of 8 so JAX doesn't recompile for every size).
- Every network sees the **same images in the same order** (`DATA_SEED`); the seed only changes the starting weights.
- `fitness` = validation accuracy after `budget` images.
- `save_params=True` also returns the trained weights (needed for the robustness test).

In [ ]:
# TRAINING
MAX_WIDTH = SEARCH_SPACE["width"][2]
GROUP = 8

def init_params(key, width):
    """Initialize the weights and biases of the neural network."""

    # Create a separate random key for each layer
    keys = jax.random.split(key, 4)

    # Initialize weights with random values scaled by the input size
    def init_weights(key, input_size, output_size, fan_in):
        weights = jax.random.normal(key, (input_size, output_size))
        return weights / jnp.sqrt(fan_in)

    # Create weights and biases for each layer
    params = {
        "W1": init_weights(keys[0], 784, MAX_WIDTH, 784),
        "b1": jnp.zeros(MAX_WIDTH),

        "W2": init_weights(keys[1], MAX_WIDTH, MAX_WIDTH, width),
        "b2": jnp.zeros(MAX_WIDTH),

        "W3": init_weights(keys[2], MAX_WIDTH, MAX_WIDTH, width),
        "b3": jnp.zeros(MAX_WIDTH),

        "Wo": init_weights(keys[3], MAX_WIDTH, 10, width),
        "bo": jnp.zeros(10)
    }

    return params

def forward(params, config, images):
    """Run the images through the neural network."""

    # Mask neurons that are outside the selected layer width
    mask = (jnp.arange(MAX_WIDTH) < config["width"]).astype(jnp.float32)

    # Choosactivation
    def activate(values):
        return jnp.where(
            config["tanh"],
            jnp.tanh(values),
            jax.nn.relu(values)
        )

    # First hidden layer
    hidden = activate(images @ params["W1"] + params["b1"])
    hidden = hidden * mask

    # Second hidden layer
    second_layer = activate(hidden @ params["W2"] + params["b2"])
    second_layer = second_layer * mask
    hidden = jnp.where(config["layers"] >= 2, second_layer, hidden)

    # Third hidden layer
    third_layer = activate(hidden @ params["W3"] + params["b3"])
    third_layer = third_layer * mask
    hidden = jnp.where(config["layers"] >= 3, third_layer, hidden)

    # Output layer produces scores for the 10 clothing categories
    output = hidden @ params["Wo"] + params["bo"]

    return output

def accuracy(p, cfg, x, y):
    return jnp.mean(jnp.argmax(forward(p, cfg, x), axis=1) == y)

def train_net(config, key, batches, train_images, train_labels, val_images, val_labels, test_images, test_labels):
    """Train one neural network and evaluate its accuracy."""

    # network's weights and biases
    params = init_params(key, config["width"])

    # optimizers
    adam = optax.adam(config["lr"])
    sgd = optax.sgd(config["lr"], momentum=SGD_MOMENTUM)
    adam_state = adam.init(params)
    sgd_state = sgd.init(params)

    # the loss
    def loss_fn(params, images, labels):
        predictions = forward(params, config, images)
        return optax.softmax_cross_entropy_with_integer_labels(predictions, labels).mean()

    # Train on one batch
    def train_step(carry, batch_indices):
        params, adam_state, sgd_state = carry

        images = train_images[batch_indices]
        labels = train_labels[batch_indices]

        gradients = jax.grad(loss_fn)(params, images, labels)

        adam_updates, adam_state = adam.update(gradients, adam_state, params)
        sgd_updates, sgd_state = sgd.update(gradients, sgd_state, params)

        # Select the correct optimizer
        updates = jax.tree_util.tree_map(
            lambda a, b: jnp.where(config["adam"], a, b),
            adam_updates, sgd_updates
        )

        params = optax.apply_updates(params, updates)
        return (params, adam_state, sgd_state), None

    # Train on all batches and evaluate after each group
    def train_group(carry, batch_group):
        carry, _ = jax.lax.scan(train_step, carry, batch_group)
        params = carry[0]
        val_accuracy = accuracy(params, config, val_images, val_labels)
        return carry, val_accuracy

    # Store validation accuracy during training
    initial_state = (params, adam_state, sgd_state)
    final_state, learning_curve = jax.lax.scan(train_group, initial_state, batches)
    params = final_state[0]

    # Evaluate the final network
    test_accuracy = accuracy(params, config, test_images, test_labels)

    return learning_curve, test_accuracy, params

# Normal training without saving weights
train_many = jax.jit(jax.vmap(lambda *args: train_net(*args)[:2],in_axes=(0, 0, None, None, None, None, None, None, None)))

# Final training also returns the trained weights
train_many_with_params = jax.jit(jax.vmap(train_net,in_axes=(0, 0, None, None, None, None, None, None, None)))

def train_population(hps, budget, seed, eval_every=None, save_params=False):
    """Train multiple networks in parallel and return their accuracies."""

    # Prepare the training budget and image order
    eval_every = eval_every or budget
    budget = (budget // eval_every) * eval_every

    #different training seeds change the network initialization but not the image order
    image_order = sample_order(budget)
    keys = jax.random.split(jax.random.PRNGKey(seed), len(hps))

    # Store results
    val = np.zeros(len(hps))
    test = np.zeros(len(hps))
    curves = [None] * len(hps)
    saved_params = [None] * len(hps)

    #use the same training batches
    batch_sizes = sorted({hp["batch_size"] for hp in hps})

    for batch_size in batch_sizes:

        batches = jnp.asarray(image_order.reshape(budget // eval_every, eval_every // batch_size,batch_size))
        # networks that use this batch size
        network_ids = [i for i, hp in enumerate(hps) if hp["batch_size"] == batch_size]
        # Train networks in smaller groups to avoid GPU memory issues
        for start in range(0, len(network_ids), MAX_PARALLEL):
            group_ids = network_ids[start:start + MAX_PARALLEL]
            # Pad the group to a multiple of GROUP for parallel training
            group_size = ((len(group_ids) + GROUP - 1) // GROUP) * GROUP
            group_size = min(MAX_PARALLEL, group_size)
            padded_ids = group_ids + [group_ids[-1]] * (group_size - len(group_ids))
            # Convert hyperparameters into JAX arrays
            config = {
              "lr": jnp.array([hps[i]["lr"] for i in padded_ids], dtype=jnp.float32),
              "adam": jnp.array([hps[i]["optimizer"] == "adam" for i in padded_ids]),
              "layers": jnp.array([hps[i]["layers"] for i in padded_ids]),
              "width": jnp.array([hps[i]["width"] for i in padded_ids]),
              "tanh": jnp.array([hps[i]["activation"] == "tanh" for i in padded_ids])
            }

            # Train the group of networks in parallel
            if save_params:
                group_curves, group_test, group_params = train_many_with_params(config, keys[jnp.array(padded_ids)], batches,x_train, y_train, x_val, y_val, x_test, y_test)
            else:
                group_curves, group_test = train_many(config, keys[jnp.array(padded_ids)], batches,x_train, y_train, x_val, y_val, x_test, y_test)

            # Convert JAX results to NumPy
            group_curves = np.asarray(group_curves)
            group_test = np.asarray(group_test)
            # Save results in the original network order
            for position, network_id in enumerate(group_ids):
                curves[network_id] = [float(accuracy) for accuracy in group_curves[position]]
                val[network_id] = group_curves[position, -1]
                test[network_id] = group_test[position]
                if save_params:
                    saved_params[network_id] = jax.tree.map(lambda weights: weights[position], group_params)

    if save_params:
        return val, curves, test, saved_params

    return val, curves, test

def train_one(hp, budget, seed, eval_every=None):
    """Train one network. Returns (final val accuracy, learning curve)."""
    val, curves, _ = train_population([hp], budget, seed, eval_every)
    return float(val[0]), (curves[0] if eval_every else [])

def fitness(hps, budget, seed):
    """FITNESS = validation accuracy after `budget` training images, for each network."""
    return train_population(hps, budget, seed)[0]

In [ ]:
# Qucik model check
hp = BASELINES["Adam 1e-3"]
acc, curve = train_one(hp, budget=12_800, seed=0, eval_every=EVAL_EVERY)
samples = np.arange(1, len(curve) + 1) * EVAL_EVERY
plt.plot(samples, curve, marker="o"); plt.grid(True)
plt.xlabel("Training samples seen"); plt.ylabel("Validation accuracy")
plt.title(f"Baseline: {describe(hp)}"); plt.show()
print(f"Final validation accuracy: {acc:.2%}")

# **Search implementation**
Each method only does two things: `propose(n)` new genomes, and `update` itself with their fitness.
- **GA**: parents by tournament (pick 3 at random, keep the best) → uniform crossover → mutation; best of parents + children survive
- **MAP-Elites**: keeps the best network per grid cell and breeds from random elites → stays diverse
- **Optuna TPE**: statistical model that suggests promising hyperparameters (searches the real values, then encodes them to a genome)
- **Random search**: random genomes, the control group

In [ ]:
def mutate(genomes, rng, sigma=0.1):
    """add small random noise to every gene"""
    return np.clip(genomes + sigma * rng.normal(size=genomes.shape), 0.0, 1.0)

def crossover(a, b, rng):
    """Uniform crossover: every gene comes from parent a or parent b with 50% chance."""
    return np.where(rng.random(a.shape) < 0.5, a, b)

def tournament_selection(pop, fit, n, rng, k=3):
  """Pick n parents, each time take k random networks and keep the best"""
  picks = rng.integers(len(pop), size=(n, k))
  best = picks[np.arange(n), np.argmax(fit[picks], axis=1)]
  return pop[best]

class RandomSearch:
    """Baseline search: random genomes."""
    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)

    def propose(self, n):
        return self.rng.random((n, N_GENES))

    def update(self, genomes, fit):
        pass

class GA:
    """Genetic algorithm: parent selection based on accuracy, crossover and mutation"""
    def __init__(self, seed):
      self.rng = np.random.default_rng(seed)
      self.pop = None # Population
      self.fit = None # Fitness

    def propose(self, n):
      # First generation is random
      if self.pop is None:
        return self.rng.random((n, N_GENES))

      # Tournament selection
      parents_a = tournament_selection(self.pop, self.fit, n, self.rng)
      parents_b = tournament_selection(self.pop, self.fit, n, self.rng)
      return mutate(crossover(parents_a, parents_b, self.rng), self.rng)

    def update(self, genomes, fit):
      if self.pop is None:
        self.pop, self.fit = np.array(genomes), np.array(fit)
        return

      # Competition between parent and children
      all_g = np.concatenate([self.pop, genomes])
      all_f = np.concatenate([self.fit, fit])

      # Keep the best candidates
      keep = np.argsort(all_f)[::-1][:len(genomes)]
      self.pop, self.fit = all_g[keep], all_f[keep]

class MAPElites:
    """Keeps the best network in every cell of the grid. New candidates are mutated children of random best networks."""
    def __init__(self, seed):
      self.rng = np.random.default_rng(seed)
      self.archive = {}

    def propose(self, n):
      # First generation is random
      if not self.archive:
            return self.rng.random((n, N_GENES))

      elites = np.array([g for g, f in self.archive.values()])
      parents_a = elites[self.rng.integers(len(elites), size=n)]
      parents_b = elites[self.rng.integers(len(elites), size=n)]
      return mutate(crossover(parents_a, parents_b, self.rng), self.rng)

    def update(self, genomes, fit):
      for g, f in zip(genomes, fit):
        cell = grid_cell(decode(g))
        if cell not in self.archive or f > self.archive[cell][1]:
          self.archive[cell] = (g, f)

class OptunaTPE:
    """A model based optimizer. Suggests genomes that its model thinks will score well."""
    def __init__(self, seed):
      self.study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
      self.trials = []

    def propose(self, n):
      self.trials = [self.study.ask() for _ in range(n)]
      hps = [{
          "lr": t.suggest_float("lr", 1e-4, 1.0, log=True),
          "optimizer": t.suggest_categorical("optimizer", ["sgd", "adam"]),
          "batch_size": t.suggest_categorical("batch_size", BATCH_SIZES),
          "layers": t.suggest_int("layers", 1, 3),
          "width": t.suggest_int("width", 16, 256),
          "activation": t.suggest_categorical("activation", ["relu", "tanh"])
      } for t in self.trials]
      return np.array([encode(hp) for hp in hps])

    def update(self, genomes, fit):
      for t, f in zip(self.trials, fit):
            self.study.tell(t, float(np.nan_to_num(f)))

SEARCH_CLASSES = {"GA": GA, "MAP-Elites": MAPElites, "Optuna TPE": OptunaTPE, "Random search": RandomSearch}

# **Performance benchmark** (step 1)
Not part of the results. Trains the same network copied *n* times in **one** GPU call, for n = 24, 48, ... 768,
and measures time and memory (first call = compile, second call = timed).
- "ms per network" drops while the GPU still has room → more parallel is free
- when it stops dropping, the GPU is full → that's the best `MAX_PARALLEL`
- out of memory → stops; use one step below

In [ ]:
# NEW: only runs when RUN_BENCHMARK = True
if RUN_BENCHMARK:
    import time

    def benchmark_group(n, budget=12_800, batch_size=32):
        global MAX_PARALLEL
        old, MAX_PARALLEL = MAX_PARALLEL, n              # train n networks in one GPU call
        hps = [dict(BASELINES["Adam 1e-3"], batch_size=batch_size)] * n
        try:
            train_population(hps, budget, seed=0)          # first call compiles for this size
            t = time.perf_counter()
            train_population(hps, budget, seed=0)          # second call = real training time
            return time.perf_counter() - t
        finally:
            MAX_PARALLEL = old

    for n in [24, 48, 96, 192, 384, 768]:
        try:
            dt = benchmark_group(n)
        except Exception as e:                             # out of memory -> stop
            print(n, "failed:", str(e)[:120])
            break
        stats = jax.devices()[0].memory_stats()
        peak = stats.get("peak_bytes_in_use", 0) / 1e9 if stats else None

        if peak is not None:
            print(f"{n:4d} networks: {dt:6.2f} s, {dt / n * 1000:6.1f} ms per network, peak memory {peak:.1f} GB")
        else:
            print(f"{n:4d} networks: {dt:6.2f} s, {dt / n * 1000:6.1f} ms per network")

# **Experimental run**
For every budget, **all searches run side by side** (4 methods x all seeds): each generation, every search proposes
`CPG` networks and they are all trained together - this keeps the GPU busy.
After the last generation, each search re-checks its `TOP_K` best on fresh seeds; the best average is its **winner**.

Result: `runs` - one entry per search (all genomes tried, their fitness, the winner).

In [ ]:
def run_search(budget):
    """Run all searches for one budge at the same time."""
    searches = []
    for name, Method in SEARCH_CLASSES.items():
        for seed in SEEDS:
            searches.append({"method": name, "seed": seed, "search": Method(seed), "genomes": [], "fitness": []})

    for gen in range(GENERATIONS):
        # picks 24 candidates
        genomes = []
        hps = []
        for s in searches:
            G = s["search"].propose(CPG)
            genomes.append(G)
            hps += [decode(g) for g in G]

        # train + measure
        fit = fitness(hps, budget, seed=gen)

         # learn from own search results
        for i, s in enumerate(searches):
            f = fit[i * CPG:(i + 1) * CPG]
            s["search"].update(genomes[i], f)
            s["genomes"] += list(genomes[i]); s["fitness"] += list(f)
    return searches

runs = []
t0 = time.time()
for budget in BUDGETS:
    t = time.time()
    for s in run_search(budget):
        G, F = np.array(s["genomes"]), np.array(s["fitness"])

        # re-check the TOP_K best with new seeds; the best average is the winner
        top = G[np.argsort(F)[::-1][:TOP_K]]
        top_hps = [decode(g) for g in top]
        recheck = np.mean([fitness(top_hps, budget, seed=1000 + 10 * s["seed"] + r)
                           for r in range(RECHECK_SEEDS)], axis=0)
        winner = top[np.argmax(recheck)]

        runs.append(dict(budget=budget, method=s["method"], seed=s["seed"], genomes=G, fitness=F,
                         winner=winner, winner_recheck=float(recheck.max())))
        print(f"budget={budget:>7} {s['method']:<14} seed {s['seed']}: best {F.max():.4f} "
              f"rechecked {recheck.max():.4f}  {describe(decode(winner))}")
    print(f"Budget {budget:,} done [{time.time() - t:.0f}s]")
print(f"All searches done in {time.time() - t0:.0f}s")

# **Retrain winners**
All winners + the baselines are trained again from scratch, longer (`EPOCHS`), `FINAL_SEEDS` times each.
Validation accuracy is measured every `EVAL_EVERY` images → learning curves. The trained weights are kept for the robustness test.

Result: `networks` - one entry per retrained network (curves, test accuracy).

In [ ]:
# RETRAIN WINNER
FINAL_SEEDS = 3

# Networks to retrain: the baselines + one winner from every search run
networks = []
for name, hp in BASELINES.items():
    networks.append({"name": "Baseline " + name, "method": "Baseline", "budget": None, "hp": hp})
for run in runs:
    name = f"{run['method']} B={run['budget']} s{run['seed']}"
    networks.append({"name": name, "method": run["method"], "budget": run["budget"], "hp": decode(run["winner"])})

all_hps = [net["hp"] for net in networks]
all_curves = []
all_test_accuracies = []
all_params = []

t0 = time.time()
for s in range(FINAL_SEEDS):
    _, curves, test_accuracies, params = train_population(
        all_hps, EPOCHS * N_TRAIN, seed=500 + s,
        eval_every=EVAL_EVERY, save_params=True)

    all_curves.append(curves)
    all_test_accuracies.append(test_accuracies)
    all_params.append(params)
    print(f"Seed {s + 1}/{FINAL_SEEDS} done [{time.time() - t0:.0f}s]")

# Store individual learning curves and their average
for i, net in enumerate(networks):
    net["seed_curves"] = np.array([all_curves[s][i] for s in range(FINAL_SEEDS)])
    net["curve"] = net["seed_curves"].mean(axis=0)

    net["test_accuracy"] = np.mean([all_test_accuracies[s][i] for s in range(FINAL_SEEDS)])

# **Robustness test**
The retrained networks are tested on **corrupted** test images (Fashion-MNIST-C: noise, blur, etc.).
These images are *never* used for training - only for this test.
`corrupted_accuracy` = accuracy on them, `accuracy_drop` = clean test accuracy - corrupted accuracy.

CHANGED: moved here (before saving), because it needs the trained weights, which are not saved.
On Rocket download the files once on the login node first (see ROCKET.md).

In [ ]:
# Load Corrupted Fashion-MNIST
import io

base_url = "https://raw.githubusercontent.com/testingautomated-usi/fashion-mnist-c/v1.0.0/generated/npy/"

def load_npy(filename):
    local = os.path.join("../data/fmnist-c", filename)        # CHANGED: use the local copy on Rocket if it exists
    if os.path.exists(local):
        return np.load(local)
    import requests
    response = requests.get(base_url + filename, timeout=120)
    response.raise_for_status()
    return np.load(io.BytesIO(response.content))

# Load corrupted test images and labels
images = load_npy("fmnist-c-test.npy")
labels = load_npy("fmnist-c-test-labels.npy")

corrupted_images = jnp.asarray(images.reshape(-1, 784) / 255.0, dtype=jnp.float32)
corrupted_labels = jnp.asarray(labels)

print("Corrupted images:", corrupted_images.shape)
print("Corrupted labels:", corrupted_labels.shape)

In [ ]:
# Evaluate robustness of all networks
for i, net in enumerate(networks):
    hp = net["hp"]

    config = {
        "lr": jnp.array(hp["lr"]),
        "adam": jnp.array(hp["optimizer"] == "adam"),
        "layers": jnp.array(hp["layers"]),
        "width": jnp.array(hp["width"]),
        "tanh": jnp.array(hp["activation"] == "tanh")
    }

    corrupted_scores = []

    for s in range(FINAL_SEEDS):
        params = all_params[s][i]
        correct = 0

        for start in range(0, len(corrupted_images), 512):
            images = corrupted_images[start:start + 512]
            labels = corrupted_labels[start:start + 512]

            predictions = forward(params, config, images)
            correct += int(jnp.sum(jnp.argmax(predictions, axis=1) == labels))

        corrupted_scores.append(correct / len(corrupted_images))

    net["corrupted_accuracy"] = np.mean(corrupted_scores)
    net["accuracy_drop"] = net["test_accuracy"] - net["corrupted_accuracy"]

# **Save results**
NEW. One file with everything the analysis needs: all searches (`runs`), the retrained networks (`networks`,
incl. robustness) and the settings used. Download it and share it with the team.

In [ ]:
# NEW
mode = "test" if TEST_RUN else ("quick" if QUICK else "full")
results_file = f"results_{mode}_{time.strftime('%Y%m%d-%H%M')}.json"
settings = dict(MODE=mode, BUDGETS=BUDGETS, SEEDS=SEEDS, GENERATIONS=GENERATIONS, CPG=CPG, EPOCHS=EPOCHS,
                TOP_K=TOP_K, RECHECK_SEEDS=RECHECK_SEEDS, FINAL_SEEDS=FINAL_SEEDS, EVAL_EVERY=EVAL_EVERY,
                THRESHOLD=THRESHOLD, SEARCH_SPACE=SEARCH_SPACE, BASELINES=BASELINES)

def to_plain(obj):
    """numpy -> plain Python, so the JSON opens anywhere (any numpy version)."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(type(obj))

with open(results_file, "w") as f:
    json.dump(dict(runs=runs, networks=networks, settings=settings), f, default=to_plain)
print(f"Saved {results_file} ({os.path.getsize(results_file) / 1e6:.1f} MB)")

# **Load results**
NEW. Analysis only: set `LOAD_RESULTS` in Setup, run **Setup**, then this cell, then everything below.
It also restores the settings of the saved run (budgets, seeds, ...), so the analysis matches it.

In [ ]:
# NEW
if LOAD_RESULTS:
    with open(LOAD_RESULTS) as f:
        saved = json.load(f)
    runs, networks, s = saved["runs"], saved["networks"], saved["settings"]
    for run in runs:
        for key in ("genomes", "fitness", "winner"):
            run[key] = np.array(run[key])
    for net in networks:
        net["seed_curves"], net["curve"] = np.array(net["seed_curves"]), np.array(net["curve"])
    BUDGETS, SEEDS, GENERATIONS, CPG = s["BUDGETS"], s["SEEDS"], s["GENERATIONS"], s["CPG"]
    EPOCHS, EVAL_EVERY, THRESHOLD, FINAL_SEEDS = s["EPOCHS"], s["EVAL_EVERY"], s["THRESHOLD"], s["FINAL_SEEDS"]
    print(f"Loaded {LOAD_RESULTS}: {s['MODE']} run, {len(runs)} searches, {len(networks)} retrained networks")
else:
    print("LOAD_RESULTS is None -> using the results of this session")

# **Evaluation**
From here on: **analysis only**, nothing is trained.

**Speed** = training images a network needs until its validation accuracy reaches 75 / 80 / 85%.
To ignore one lucky spike, the average of 5 consecutive checks must reach the threshold. `never` = it didn't.

In [ ]:
# Speed - number of training images a network needs to reach 75%, 80% and 85% accuracy.

def images_to_reach(curve, threshold, k=5):
    """Images seen until the average of k checks in a row first reaches the threshold """
    smooth = np.convolve(curve, np.ones(k) / k, mode="valid")
    hits = np.nonzero(smooth >= threshold)[0]
    return (hits[0] + k) * EVAL_EVERY if len(hits) else np.inf

# Calculate learning speed for each seed separately
for net in networks:
    net["seed_speeds"] = {}
    net["speed"] = {}

    for threshold in THRESHOLD:
        # Calculate how many images each seed needs
        speeds = []

        for curve in net["seed_curves"]:
            images = images_to_reach(curve, threshold)
            speeds.append(images)

        # Store individual results and their average
        net["seed_speeds"][threshold] = speeds
        net["speed"][threshold] = np.mean(speeds)


baselines = [net for net in networks if net["method"] == "Baseline"]
winners = [net for net in networks if net["method"] != "Baseline"]

# Table: images needed by every network
print(f"{'Network':<28} {'to 75%':>8} {'to 80%':>8} {'to 85%':>8}   Hyperparameters")
for net in networks:
    speeds = " ".join(f"{s:>8,.0f}" if s < np.inf else f"{'never':>8}" for s in net["speed"].values())
    print(f"{net['name']:<28} {speeds}   {describe(net['hp'])}")

**Search methods vs random search**: the winner's accuracy (re-checked) per budget and seed, and a Wilcoxon test against random search.

In [ ]:
import pandas as pd

# Collect the winning accuracy from each experiment
results = []

for run in runs:
    results.append({
        "Budget": run["budget"],
        "Method": run["method"],
        "Seed": run["seed"],
        "Accuracy": run["winner_recheck"]
    })

results_df = pd.DataFrame(results)

# Compare methods using the same budget and seed
comparison = results_df.pivot(
    index=["Budget", "Seed"],
    columns="Method",
    values="Accuracy"
)

display(comparison)

In [ ]:
from scipy.stats import wilcoxon

# Compare each method against Random Search
for budget in BUDGETS:
    data = comparison.loc[budget]

    for method in ["GA", "MAP-Elites", "Optuna TPE"]:
        differences = data[method] - data["Random search"]

        # Test whether the methods have different accuracies
        if np.allclose(differences, 0):
            p_value = 1.0
        else:
            p_value = wilcoxon(differences).pvalue

        print(f"Budget {budget}, {method}: p = {p_value:.4f}")

# **Evolution**
How the searches improve over the generations (first budget, first seed): average accuracy per generation,
the best network found so far, and for MAP-Elites the diversity grid, its coverage and QD-score.

In [ ]:
# Compare how the search methods improve over generations
plt.figure(figsize=(10, 5))

for run in runs:
    if run["budget"] != BUDGETS[0] or run["seed"] != SEEDS[0]:
        continue

    # Get the accuracy of all networks in each generation
    scores = run["fitness"].reshape(GENERATIONS, CPG)
    # Calculate the average accuracy for each generation
    average_accuracy = scores.mean(axis=1)

    plt.plot(range(1, GENERATIONS + 1), average_accuracy, marker="o", label=run["method"])

plt.xlabel("Generation")
plt.ylabel("Average validation accuracy")
plt.title("Evolution of validation accuracy")
plt.xticks(range(1, GENERATIONS + 1))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Compare the best networks found over generations
plt.figure(figsize=(10, 5))

for run in runs:
    if run["budget"] != BUDGETS[0] or run["seed"] != SEEDS[0]:
        continue

    scores = run["fitness"].reshape(GENERATIONS, CPG)
    # Best accuracy in each generation
    best = scores.max(axis=1)
    # Keep the best accuracy found so far
    best_so_far = np.maximum.accumulate(best)
    plt.plot(range(1, GENERATIONS + 1), best_so_far, marker="o", label=run["method"])

plt.xlabel("Generation")
plt.ylabel("Best validation accuracy")
plt.title("Best accuracy found over generations")
plt.xticks(range(1, GENERATIONS + 1))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Select MAP-Elites results
map_run = next(run for run in runs if run["method"] == "MAP-Elites")

# Create an empty diversity grid
grid = np.full((GRID_BINS, GRID_BINS), np.nan)

# Store the best accuracy found in each cell
for genome, score in zip(map_run["genomes"], map_run["fitness"]):
    row, col = grid_cell(decode(genome))

    if np.isnan(grid[row, col]) or score > grid[row, col]:
        grid[row, col] = score

# Display the grid
plt.figure(figsize=(8, 6))
plt.imshow(grid, origin="lower", vmin=0, vmax=1)
plt.colorbar(label="Validation accuracy")

# Real values in the middle of each grid cell
lr_values = np.logspace(-4, 0, 2 * GRID_BINS + 1)[1::2]
size_values = np.logspace(np.log10(_P_MIN), np.log10(_P_MAX), 2 * GRID_BINS + 1)[1::2]
plt.yticks(range(GRID_BINS), [f"{v:.4f}" for v in lr_values])
plt.xticks(range(GRID_BINS), [f"{v / 1000:.0f}k" for v in size_values], rotation=45)
plt.xlabel("Number of parameters")
plt.ylabel("Learning rate")
plt.title("MAP-Elites diversity grid")
plt.show()

print("Occupied cells:", np.count_nonzero(~np.isnan(grid)))

In [ ]:
occupied = np.count_nonzero(~np.isnan(grid))
coverage = occupied / grid.size

print(f"Grid coverage: {coverage:.1%}")

# Sum the best accuracies of all occupied grid cells
qd_score = np.nansum(grid)
print(f"QD-score: {qd_score:.4f}")

In [ ]:
# Track MAP-Elites grid coverage over generations
occupied_cells = set()
coverage = []

for genome in map_run["genomes"]:
    occupied_cells.add(grid_cell(decode(genome)))
    coverage.append(len(occupied_cells) / GRID_BINS**2)

# Plot coverage after each generation
coverage_per_generation = np.array(coverage)[CPG - 1::CPG]

plt.plot(range(1, GENERATIONS + 1), coverage_per_generation, marker="o")
plt.xlabel("Generation")
plt.ylabel("Grid coverage")
plt.title("MAP-Elites diversity over generations")
plt.grid(True)
plt.show()

# **Analysis**
- **H1 - fast learners:** how many winners beat the best baseline? Does a smaller search budget give faster learners (Spearman)?
- **Trade-off:** final validation / test accuracy of the winners vs the baselines, and speed vs final accuracy.
- **Diversity (MAP-Elites vs the others):** all networks of every search go into the same grid;
  MAP-Elites is compared with each method on QD-score and good cells (Mann-Whitney).

In [ ]:
def fast_learner_test(group, threshold):
    """Count how many winners learn faster than the best baseline."""

    best_baseline = min(b["speed"][threshold] for b in baselines)

    faster_winners = 0
    for winner in group:
        if winner["speed"][threshold] < best_baseline:
            faster_winners += 1

    return faster_winners

for threshold in THRESHOLD:
    best_baseline = min(b["speed"][threshold] for b in baselines)
    print(f"\nReach {threshold:.0%}: best baseline needs {best_baseline:,.0f} images")

    for budget in BUDGETS:
        for method in ["GA", "MAP-Elites", "Optuna TPE", "Random search"]:
            group = [w for w in winners if w["budget"] == budget and w["method"] == method]

            faster = fast_learner_test(group, threshold)
            print(f"  {method:<14} budget {budget:>7,}: {faster}/{len(group)} faster")

for th in THRESHOLD:
    reached = [w for w in winners if np.isfinite(w["speed"][th])]
    rho, p = spearmanr([w["budget"] for w in reached], [w["speed"][th] for w in reached])
    print(f"\nReach {th:.0%}: rho = {rho:.2f}, p = {p:.4f}  (positive = smaller budget gives faster learners)")
    for B in BUDGETS:
        speeds = [w["speed"][th] for w in winners if w["budget"] == B]
        print(f"  budget {B:>7,}: median {np.median(speeds):,.0f} images")

#  Plots
images = np.arange(1, len(networks[0]["curve"]) + 1) * EVAL_EVERY
first_epoch = images <= N_TRAIN
budget_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
baseline_styles = ["--", ":", "-."]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot
for B, color in zip(BUDGETS, budget_colors):
    mean_curve = np.mean([w["curve"] for w in winners if w["budget"] == B], axis=0)
    ax1.plot(images[first_epoch], mean_curve[first_epoch], color=color, linewidth=2.5,
             label=f"Winners, budget {B:,} (average)")
for b, style in zip(baselines, baseline_styles):
    ax1.plot(images[first_epoch], b["curve"][first_epoch], color="gray", linestyle=style, linewidth=2,
             label=b["name"])
for th in THRESHOLD:
    ax1.axhline(th, color="black", linewidth=0.8)
    ax1.text(images[0], th + 0.003, f"{th:.0%}", fontsize=9)
ax1.set_xscale("log")
ax1.set_ylim(0.5, 0.9)
ax1.set_xlabel("Training images seen (log scale)")
ax1.set_ylabel("Validation accuracy")
ax1.set_title("Learning curves in the first epoch")
ax1.legend(fontsize=8, loc="lower right")

# Right plot
for b, marker in zip(baselines, ["s", "^", "D"]):
    ax2.scatter(0, b["speed"][0.80], color="gray", marker=marker, s=80, label=b["name"])
for i, (B, color) in enumerate(zip(BUDGETS, budget_colors), start=1):
    speeds = [w["speed"][0.80] for w in winners if w["budget"] == B and w["speed"][0.80] < np.inf]
    x = i + np.random.uniform(-0.15, 0.15, len(speeds))
    ax2.scatter(x, speeds, color=color)
ax2.set_xticks(range(len(BUDGETS) + 1))
ax2.set_xticklabels(["Baselines"] + [f"Winners\nbudget {B:,}" for B in BUDGETS])
ax2.set_ylim(0, None)
ax2.set_ylabel("Images needed to reach 80% (lower = faster)")
ax2.set_title("Images needed to reach 80% (one dot = one network)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("h1_results.png", dpi=150)
plt.show()

In [ ]:
# Compare the final validation accuracy of all networks
results = []

for net in networks:
    final_accuracies = net["seed_curves"][:, -1]

    results.append({
        "Network": net["name"],
        "Method": net["method"],
        "Budget": net["budget"],
        "Mean accuracy": np.mean(final_accuracies),
        "Std accuracy": np.std(final_accuracies)
    })

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Compare final validation accuracy across search methods
summary = results_df.groupby("Method")["Mean accuracy"].agg(["mean", "std"])
summary = summary * 100
display(summary.round(2))

In [ ]:
# Compare learning speed with final validation accuracy
plt.figure(figsize=(9, 5))

for method in sorted(set(net["method"] for net in networks)):
    group = [net for net in networks if net["method"] == method]
    speeds = []
    accuracies = []

    for net in group:
        speed = net["speed"][0.80]

        if np.isfinite(speed):
            speeds.append(speed)
            accuracies.append(net["curve"][-1])

    plt.scatter(speeds, accuracies, label=method, s=60, alpha=0.7)

plt.xlabel("Images needed to reach 80% accuracy")
plt.ylabel("Final validation accuracy")
plt.title("Learning speed vs final accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Compare final test accuracy across search methods
test_df = pd.DataFrame(networks)
summary = test_df.groupby("method")["test_accuracy"].agg(["mean", "std"])

display((summary * 100).round(2))

In [ ]:
# H2: diversity of every search, MAP-Elites vs the others
from scipy.stats import mannwhitneyu

def archive_of(run):
    """Best accuracy in every grid cell that this search reached."""
    archive = {}
    for genome, score in zip(run["genomes"], run["fitness"]):
        cell = grid_cell(decode(genome))
        archive[cell] = max(archive.get(cell, 0), score)
    return archive

for run in runs:
    archive = archive_of(run)
    best = max(r["fitness"].max() for r in runs if r["budget"] == run["budget"])   # best network of this budget
    run["qd_score"] = sum(archive.values())
    run["good_cells"] = sum(score >= best - GOOD_CELL_MARGIN for score in archive.values())

for budget in BUDGETS:
    print(f"\nBudget {budget:,}")
    map_elites = [r for r in runs if r["budget"] == budget and r["method"] == "MAP-Elites"]
    for other in ["GA", "Optuna TPE", "Random search"]:
        others = [r for r in runs if r["budget"] == budget and r["method"] == other]
        for key in ["qd_score", "good_cells"]:
            a = [r[key] for r in map_elites]
            b = [r[key] for r in others]
            p = mannwhitneyu(a, b, alternative="greater").pvalue
            print(f"  {key:<10}: MAP-Elites {np.mean(a):5.1f} vs {other:<13} {np.mean(b):5.1f}   p = {p:.4f}")

# **Robustness analysis**
Uses `corrupted_accuracy` / `accuracy_drop` from the robustness test above.
Are fast learners less robust? Spearman correlation between speed and corrupted accuracy / accuracy drop.

In [ ]:
# Compare clean and corrupted test accuracy
import pandas as pd

robustness_df = pd.DataFrame([
    {
        "Network": net["name"],
        "Clean accuracy (%)": net["test_accuracy"] * 100,
        "Corrupted accuracy (%)": net["corrupted_accuracy"] * 100,
        "Accuracy drop (pp)": net["accuracy_drop"] * 100
    }
    for net in networks
])

display(robustness_df.round(2))

In [ ]:
# Learning speed vs robustness
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

robustness = np.array([net["corrupted_accuracy"] for net in networks])

for threshold in [0.75, 0.80, 0.85]:
    speeds = np.array([net["speed"][threshold] for net in networks])
    valid = np.isfinite(speeds)

    if valid.sum() < 3:
        print(f"{threshold:.0%}: Not enough networks ({valid.sum()}/{len(networks)})")
        continue

    rho, p = spearmanr(speeds[valid], robustness[valid])

    plt.figure(figsize=(7, 4))
    plt.scatter(speeds[valid], robustness[valid] * 100)
    plt.xlabel(f"Training images needed to reach {threshold:.0%} validation accuracy")
    plt.ylabel("Corrupted test accuracy (%)")
    plt.title(f"Threshold {threshold:.0%} (Spearman ρ = {rho:.2f})")
    plt.grid(alpha=0.3)
    plt.show()

    print(f"{threshold:.0%}: Spearman ρ = {rho:.3f}, p = {p:.3f}, "
          f"networks = {valid.sum()}/{len(networks)}")

In [ ]:
# Learning speed vs accuracy drop
accuracy_drops = np.array([net["accuracy_drop"] for net in networks])

for threshold in [0.75, 0.80, 0.85]:
    speeds = np.array([net["speed"][threshold] for net in networks])
    valid = np.isfinite(speeds)

    if valid.sum() < 3:
        print(f"{threshold:.0%}: Not enough networks ({valid.sum()}/{len(networks)})")
        continue

    rho, p = spearmanr(speeds[valid], accuracy_drops[valid])

    plt.figure(figsize=(7, 4))
    plt.scatter(speeds[valid], accuracy_drops[valid] * 100)
    plt.xlabel(f"Training images needed to reach {threshold:.0%} validation accuracy")
    plt.ylabel("Accuracy drop (percentage points)")
    plt.title(f"Threshold {threshold:.0%} (Spearman ρ = {rho:.2f})")
    plt.grid(alpha=0.3)
    plt.show()

    print(f"{threshold:.0%}: Spearman ρ = {rho:.3f}, p = {p:.3f}, "
          f"networks = {valid.sum()}/{len(networks)}")